# 📹 Procesamiento de Video con YOLO

Este notebook está diseñado únicamente para procesar un video utilizando los 3 modelos integrados:
1. **Detección OBB** (modelo personalizado de tenis de mesa entrenado con YOLO26 OBB).
2. **Segmentación de Objetos** (`yolo26n-seg.pt` de COCO, excluyendo personas).
3. **Pose Estimation** (`yolo26n-pose.pt` para el esqueleto de los jugadores).

El procesamiento está limitado para extraer y analizar únicamente **10 segundos** del video original.

### 1. Instalación de Librerías
Instalamos los paquetes necesarios para procesar video e inferencia de modelos.

In [ ]:
%pip install ultralytics opencv-python numpy pyyaml matplotlib

### 2. Parámetros de Confianza Ajustables
Modifica los siguientes valores para calibrar la sensibilidad de cada modelo según necesites. Si bajas el valor, detectará más objetos pero con mayor probabilidad de falsos positivos.

In [ ]:
# Umbrales de confianza mínimos para cada modelo (rango de 0.0 a 1.0)
CONF_DETECTION = 0.25  # Para tu modelo personalizado de OBB (Tenis de Mesa)
CONF_SEG       = 0.30  # Para el modelo de segmentación de objetos COCO
CONF_POSE      = 0.25  # Para el modelo de pose (esqueleto de personas)

print(f"⚙️ Parámetros de confianza configurados:")
print(f"   - Detección OBB: {CONF_DETECTION}")
print(f"   - Segmentación:  {CONF_SEG}")
print(f"   - Pose:          {CONF_POSE}")

### 3. Cargar Modelos
Buscamos el mejor modelo entrenado (`best.pt`) y cargamos los modelos preentrenados de Segmentación y Pose de **YOLO26**.

In [ ]:
import os
import glob
from collections import defaultdict
import numpy as np
import cv2
from ultralytics import YOLO

# Buscar dinámicamente los mejores pesos entrenados
best_weights = glob.glob('**/weights/best.pt', recursive=True)
if best_weights:
    BEST_WEIGHTS_PATH = best_weights[0]
    print(f"🎯 Modelo personalizado cargado desde: {BEST_WEIGHTS_PATH}")
else:
    BEST_WEIGHTS_PATH = 'yolo26n-obb.pt'
    print(f"⚠️ No se encontró best.pt, usando modelo base: {BEST_WEIGHTS_PATH}")

model_detection    = YOLO(BEST_WEIGHTS_PATH)
model_segmentation = YOLO('yolo26n-seg.pt')
model_pose         = YOLO('yolo26n-pose.pt')

print("✅ Modelos cargados correctamente")

### 4. Configuración del Video de Entrada/Salida
Busca los videos disponibles en el directorio `Videos/` y configura el video resultante de 10 segundos.

In [ ]:
# Buscar videos en la carpeta Videos/
videos = glob.glob('Videos/*.mp4')

if videos:
    VIDEO_INPUT = videos[0]
    print(f"📹 Video encontrado para procesar: {VIDEO_INPUT}")
else:
    VIDEO_INPUT = "video1.mp4"
    print(f"⚠️ No se encontró la carpeta Videos/, usando fallback: {VIDEO_INPUT}")

VIDEO_OUTPUT = 'video_procesado_10s.mp4'
print(f"💾 El video procesado se guardará en: {VIDEO_OUTPUT}")

### 5. Definición de Funciones de Dibujo (Visualización)
Configuramos colores personalizados y lógica para dibujar bounding boxes orientadas (OBB), segmentación COCO (sin personas), pose estimation e información en pantalla.

In [ ]:
CUSTOM_CLASS_COLORS = {
    'tt net': (252, 191, 73),      # Amarillo/Naranja
    'tt racket': (0, 180, 216),    # Celeste
    'tt table': (255, 0, 110),     # Rosa/Fucsia
}

COCO_PERSON_CLASS_ID = 0

SKELETON_CONNECTIONS = [
    (0, 1), (0, 2), (1, 3), (2, 4),        # Cabeza
    (5, 6),                                  # Hombros
    (5, 7), (7, 9),                          # Brazo izquierdo
    (6, 8), (8, 10),                         # Brazo derecho
    (5, 11), (6, 12),                        # Torso
    (11, 12),                                # Caderas
    (11, 13), (13, 15),                      # Pierna izquierda
    (12, 14), (14, 16),                      # Pierna derecha
]
KEYPOINT_COLOR  = (0, 255, 255)    
SKELETON_COLOR  = (255, 255, 0)    
SKELETON_COLOR2 = (0, 165, 255)    

def draw_segmentation(frame, seg_results, conf_thresh):
    overlay = frame.copy()
    if seg_results.masks is None:
        return frame

    masks = seg_results.masks.data.cpu().numpy()
    boxes = seg_results.boxes
    h, w  = frame.shape[:2]

    np.random.seed(42)
    coco_palette = np.random.randint(50, 220, (80, 3), dtype=np.uint8)

    for i, (mask, box) in enumerate(zip(masks, boxes)):
        cls_id     = int(box.cls[0])
        confidence = float(box.conf[0])
        cls_name   = seg_results.names[cls_id]

        if cls_id == COCO_PERSON_CLASS_ID:
            continue
        if confidence < conf_thresh:
            continue

        mask_resized = cv2.resize(mask.astype(np.float32), (w, h), interpolation=cv2.INTER_LINEAR)
        mask_bool = mask_resized > 0.5

        color = coco_palette[cls_id % 80].tolist()
        overlay[mask_bool] = (overlay[mask_bool] * 0.45 + np.array(color) * 0.55).astype(np.uint8)

        mask_uint8 = mask_bool.astype(np.uint8) * 255
        contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(frame, contours, -1, color, 1)

        x1, y1 = int(box.xyxy[0][0]), int(box.xyxy[0][1])
        label = f"{cls_name} {confidence:.2f}"
        cv2.putText(overlay, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)

    cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)
    return frame

def draw_pose(frame, pose_results, conf_thresh):
    for person in pose_results:
        boxes = person.boxes
        kpts  = person.keypoints
        if boxes is None or kpts is None: 
            continue
        for box, kp in zip(boxes, kpts.data):
            conf_person = float(box.conf[0])
            if conf_person < conf_thresh:
                continue

            keypoints_xy = kp[:, :2].cpu().numpy()
            keypoints_c  = kp[:, 2].cpu().numpy()

            for (kp1_idx, kp2_idx) in SKELETON_CONNECTIONS:
                if kp1_idx >= len(keypoints_xy) or kp2_idx >= len(keypoints_xy):
                    continue
                if keypoints_c[kp1_idx] < 0.3 or keypoints_c[kp2_idx] < 0.3:
                    continue

                x1, y1 = int(keypoints_xy[kp1_idx][0]), int(keypoints_xy[kp1_idx][1])
                x2, y2 = int(keypoints_xy[kp2_idx][0]), int(keypoints_xy[kp2_idx][1])
                s_color = SKELETON_COLOR2 if kp1_idx in [7,8,9,10] else SKELETON_COLOR
                cv2.line(frame, (x1, y1), (x2, y2), s_color, 2, cv2.LINE_AA)

            for idx, (x, y) in enumerate(keypoints_xy):
                if keypoints_c[idx] < 0.3:
                    continue
                cv2.circle(frame, (int(x), int(y)), 4, KEYPOINT_COLOR, -1, cv2.LINE_AA)
                cv2.circle(frame, (int(x), int(y)), 4, (0, 0, 0), 1, cv2.LINE_AA)
    return frame

def draw_custom_detections(frame, det_results, conf_thresh):
    counts = defaultdict(int)

    is_obb = hasattr(det_results, 'obb') and det_results.obb is not None and len(det_results.obb) > 0
    is_boxes = hasattr(det_results, 'boxes') and det_results.boxes is not None and len(det_results.boxes) > 0

    if not is_obb and not is_boxes:
        return frame, dict(counts)

    boxes_list = det_results.obb if is_obb else det_results.boxes

    for box in boxes_list:
        cls_id     = int(box.cls[0])
        cls_name   = det_results.names[cls_id]
        confidence = float(box.conf[0])

        if confidence < conf_thresh:
            continue

        normalized_name = cls_name.lower().strip()
        color = CUSTOM_CLASS_COLORS.get(normalized_name, (200, 200, 200))
        
        display_name = 'TT Net' if normalized_name == 'tt net' else ('TT Racket' if normalized_name == 'tt racket' else ('TT Table' if normalized_name == 'tt table' else cls_name))
        counts[display_name] += 1

        if is_obb:
            pts = box.xyxyxyxy[0].cpu().numpy().astype(np.int32)
            cv2.polylines(frame, [pts], isClosed=True, color=color, thickness=2, lineType=cv2.LINE_AA)
            x1, y1 = pts[0][0], pts[0][1]
        else:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        label = f"{display_name} {confidence:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 6, y1), color, -1)
        cv2.putText(frame, label, (x1 + 3, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

    return frame, dict(counts)

def draw_info_panel(frame, counts, fps, frame_idx, n_persons):
    all_custom = ['TT Racket', 'TT Net', 'TT Table']
    panel_lines = len(all_custom) + 2
    panel_h = 30 + panel_lines * 28 + 20
    panel_w = 240

    overlay = frame.copy()
    cv2.rectangle(overlay, (8, 8), (8 + panel_w, 8 + panel_h), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.72, frame, 0.28, 0, frame)

    cv2.putText(frame, "Objetos Detectados", (18, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (220, 220, 220), 1, cv2.LINE_AA)
    cv2.line(frame, (18, 40), (8 + panel_w - 8, 40), (80, 80, 80), 1)

    y = 65
    for cls_name in all_custom:
        count = counts.get(cls_name, 0)
        color = CUSTOM_CLASS_COLORS.get(cls_name.lower().strip(), (200, 200, 200))
        cv2.circle(frame, (26, y - 5), 6, color, -1)
        cv2.putText(frame, f"{cls_name}: {count}", (40, y), cv2.FONT_HERSHEY_SIMPLEX, 0.53, (240, 240, 240), 1, cv2.LINE_AA)
        y += 28

    cv2.putText(frame, f"Personas: {n_persons}", (18, y), cv2.FONT_HERSHEY_SIMPLEX, 0.53, (0, 255, 255), 1, cv2.LINE_AA)
    y += 28
    cv2.putText(frame, f"FPS: {fps:.1f}  |  Frame: {frame_idx}", (18, y + 8), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (130, 130, 130), 1, cv2.LINE_AA)
    return frame

### 6. Pipeline de Procesamiento (Límite 10 segundos)
Corremos el pipeline que lee el video, ejecuta inferencia con los 3 modelos, ensambla el panel de visualización y guarda los primeros 10 segundos en un nuevo archivo.

In [ ]:
import time

cap = cv2.VideoCapture(VIDEO_INPUT)
if not cap.isOpened():
    raise FileNotFoundError(f"❌ No se pudo abrir: {VIDEO_INPUT}")

fps_in  = cap.get(cv2.CAP_PROP_FPS) or 30
width   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Calcular el total de frames para procesar exactamente 10 segundos
SEGUNDOS_A_PROCESAR = 10
limit_frames = int(SEGUNDOS_A_PROCESAR * fps_in)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(VIDEO_OUTPUT, fourcc, fps_in, (width, height))

frame_idx = 0
print(f"📹 Procesando {SEGUNDOS_A_PROCESAR} segundos de video ({limit_frames} frames)...\n")

try:
    while frame_idx < limit_frames:
        ret, frame = cap.read()
        if not ret:
            break

        t0 = time.time()

        # 1. Segmentación (COCO sin personas)
        seg_results = model_segmentation.predict(frame, conf=CONF_SEG, iou=0.45, imgsz=640, verbose=False)[0]
        frame = draw_segmentation(frame, seg_results, CONF_SEG)

        # 2. Pose Estimation
        pose_results = model_pose.predict(frame, conf=CONF_POSE, iou=0.45, imgsz=640, verbose=False)
        frame = draw_pose(frame, pose_results, CONF_POSE)

        # Contar personas para el panel
        n_persons = 0
        for pr in pose_results:
            if pr.boxes is not None:
                n_persons += sum(1 for c in pr.boxes.conf if float(c) >= CONF_POSE)

        # 3. Detección Personalizada (OBB o estándar)
        det_results = model_detection.predict(frame, conf=CONF_DETECTION, iou=0.45, imgsz=640, verbose=False)[0]
        frame, custom_counts = draw_custom_detections(frame, det_results, CONF_DETECTION)

        # 4. Dibujar Panel de Información
        elapsed_fps = 1.0 / (time.time() - t0 + 1e-9)
        frame = draw_info_panel(frame, custom_counts, elapsed_fps, frame_idx, n_persons)

        # Guardar frame procesado
        out.write(frame)
        frame_idx += 1

        if frame_idx % 30 == 0:
            pct = (frame_idx / limit_frames * 100)
            print(f"  Frame {frame_idx:04d}/{limit_frames} ({pct:.0f}%) — {elapsed_fps:.1f} FPS")

finally:
    cap.release()
    out.release()

print(f"\n✅ ¡Completado! Video procesado guardado en: {VIDEO_OUTPUT}")